# U-Net ile Karaciger Segmentasyonu (LiTS Dataset) - Colab Egitimi

Bu notebook, yerel makinede yazilan U-Net + Dice/BCE loss + IoU metrigi kodunu Colab GPU'sunda egitmek icin hazirlanmistir.

**Adimlar:** Kaggle anahtari yukle -> LiTS verisini indir -> 2D dilimlere ayir (preprocess) -> U-Net'i egit.

Calistirmadan once ust menuden **Runtime > Change runtime type > GPU** secili oldugundan emin ol.

## 1. Kaggle API anahtarini yukle
Asagidaki hucreyi calistirip `kaggle.json` dosyani sec (Kaggle hesabinin Account sayfasindan indirdigin dosya).

In [ ]:
from google.colab import files
uploaded = files.upload()  # kaggle.json dosyasini sec


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle nibabel opencv-python-headless


## 2. LiTS Part 1 veri setini indir ve ac
`--force`: eger bu hucreyi daha once calistirip yarim birakmissan, eski
zip dosyasi hala diskte kalir ve Kaggle CLI "uzerine yazayim mi?" diye
sorup hucrenin sonsuza kadar beklemesine (donmus gibi gorunmesine) sebep
olabilir. `--force` bu soruyu sormadan direkt uzerine yazar.


In [ ]:
!rm -f liver-tumor-segmentation.zip
!rm -rf lits_data
!kaggle datasets download -d andrewmvd/liver-tumor-segmentation --force
!mkdir -p lits_data
!unzip -q liver-tumor-segmentation.zip -d lits_data
!ls lits_data


## 3. Proje kodunu yaz (yerel projedeki src/ dosyalariyla ayni)

In [ ]:
%%writefile model.py
"""U-Net modeli (PyTorch) - sifirdan, skip connection'lar dahil."""

import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    """(Conv2d -> BatchNorm -> ReLU) iki kez ust uste.

    U-Net'in her seviyesinde (hem encoder hem decoder tarafinda) tekrar
    eden temel blok budur.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Klasik U-Net: 4 seviyeli encoder, bottleneck, 4 seviyeli decoder.

    in_channels: girdi goruntusunun kanal sayisi (CT icin genelde 1 - gri tonlama)
    out_channels: cikti maskesinin kanal sayisi (binary segmentasyon icin 1)
    features: her encoder seviyesindeki kanal sayilari
    """

    def __init__(self, in_channels=1, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ---- Encoder (daralan yol) ----
        self.encoders = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f))
            ch = f

        # ---- Bottleneck (en alt nokta) ----
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # ---- Decoder (genisleyen yol) ----
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        rev_features = list(reversed(features))
        in_ch = features[-1] * 2
        for f in rev_features:
            self.upconvs.append(
                nn.ConvTranspose2d(in_ch, f, kernel_size=2, stride=2)
            )
            # concat sonrasi kanal sayisi 2*f olur (upconv ciktisi + skip)
            self.decoders.append(DoubleConv(f * 2, f))
            in_ch = f

        # ---- Cikti katmani ----
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder: her seviyede conv uygula, sonucu sakla (skip), sonra kucult
        for encoder in self.encoders:
            x = encoder(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        # Skip'leri decoder sirasina gore kullanmak icin ters cevir
        skip_connections = skip_connections[::-1]

        # Decoder: buyut, ayni seviyedeki skip ile birlestir (concat), conv uygula
        for i in range(len(self.decoders)):
            x = self.upconvs[i](x)
            skip = skip_connections[i]

            # Boyut uyuşmazligi olursa (tek sayili genislik/yukseklik gibi
            # durumlarda cikabilir) skip'e gore kirp/hizala
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])

            x = torch.cat([skip, x], dim=1)  # kanal boyutunda birlestir
            x = self.decoders[i](x)

        return self.final_conv(x)


if __name__ == "__main__":
    # Hizli bir sanity check: rastgele bir goruntu ile boyutlarin dogru aktigini test et
    model = UNet(in_channels=1, out_channels=1)
    dummy_input = torch.randn(2, 1, 256, 256)  # (batch, channel, height, width)
    output = model(dummy_input)
    print("Girdi boyutu :", dummy_input.shape)
    print("Cikti boyutu :", output.shape)


In [ ]:
%%writefile losses.py
"""Dice + BCE combined loss ve IoU metrigi."""

import torch
import torch.nn as nn


class DiceLoss(nn.Module):
    """Dice katsayisina dayali loss.

    Dice katsayisi iki maske arasindaki ortusmeyi olcer:
        Dice = (2 * |tahmin ve gercek kesisimi|) / (|tahmin| + |gercek|)

    Dice = 1  -> mukemmel ortusme
    Dice = 0  -> hic ortusme yok

    Loss olarak kullanmak icin (1 - Dice) alinir, boylece optimizasyon
    Dice'i 1'e yaklastirmaya calisir.
    """

    def __init__(self, smooth=1e-6):
        super().__init__()
        # smooth: paydanin sifir olmasini engelleyen kucuk sabit
        # (hem tahmin hem gercek maske tamamen bos oldugunda 0/0 hatasini onler)
        self.smooth = smooth

    def forward(self, logits, targets):
        # logits: modelin ham ciktisi (sigmoid uygulanmamis), sekil [B, 1, H, W]
        # targets: gercek maske, 0/1 degerli, ayni sekil
        probs = torch.sigmoid(logits)  # once olasiliga cevir (0-1 arasi)

        probs = probs.view(-1)
        targets = targets.view(-1)  # duz vektore cevir, piksel piksel karsilastir

        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum() + targets.sum() + self.smooth
        )
        return 1.0 - dice


class DiceBCELoss(nn.Module):
    """Dice loss + Binary Cross Entropy loss toplami.

    Neden ikisi birden?
    - BCE: her pikseli tek tek, birbirinden bagimsiz degerlendirir. Egitimin
      basinda kararli ve guclu bir gradyan saglar, ama sinif dengesizligine
      (goruntude cok az karaciger pikseli, cok fazla arka plan pikseli varsa)
      duyarlidir; model kolayca "her yeri arka plan de" diyerek dusuk loss
      elde edebilir.
    - Dice: buyuk/kucuk nesne ayrimi yapmadan, ortusme oranina odaklanir. Sinif
      dengesizligine BCE kadar duyarli degildir, kucuk karaciger bolgelerinde
      de anlamli bir sinyal verir.

    Ikisini toplayarak, BCE'nin egitimi stabilize eden etkisini, Dice'in
    ortusmeye odaklanan etkisiyle birlestirmis oluyoruz.
    """

    def __init__(self, smooth=1e-6):
        super().__init__()
        self.dice = DiceLoss(smooth=smooth)
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.bce(logits, targets)


def iou_score(logits, targets, threshold=0.5, smooth=1e-6):
    """Intersection over Union (Jaccard index) metrigi.

    IoU = |tahmin ve gercek kesisimi| / |tahmin ve gercek birlesimi|

    Egitim sirasinda optimize edilen bir loss degil, modelin ne kadar iyi
    oldugunu insanin anlayacagi sekilde raporlamak icin kullanilan bir
    degerlendirme metrigidir. Bu yuzden gradyan hesaplamaya gerek yok
    (torch.no_grad).
    """
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > threshold).float()  # olasiligi kesin 0/1 maskeye cevir
        targets = targets.float()

        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()
        union = preds.sum() + targets.sum() - intersection

        iou = (intersection + smooth) / (union + smooth)
        return iou.item()


if __name__ == "__main__":
    # Sanity check: rastgele tahmin ve gercek maske ile loss/metrik hesapla
    torch.manual_seed(0)
    logits = torch.randn(2, 1, 64, 64)          # modelin ham ciktisi (sigmoid oncesi)
    targets = torch.randint(0, 2, (2, 1, 64, 64)).float()  # rastgele 0/1 gercek maske

    criterion = DiceBCELoss()
    loss = criterion(logits, targets)
    iou = iou_score(logits, targets)

    print("Loss (Dice+BCE):", loss.item())
    print("IoU             :", iou)


In [ ]:
%%writefile preprocess.py
"""LiTS NIfTI volume'larini 2D dilimlere ayirip .npy olarak diske kaydeder.

Neden bu adima ihtiyacimiz var?
- Her CT volume'u (512, 512, 75) gibi 3 boyutlu ve nibabel ile okunmasi
  goreceli yavas. Egitim sirasinda her epoch'ta ayni volume'lari tekrar
  tekrar NIfTI'den okumak yerine, tek seferde 2D dilimlere bolup hafif
  .npy dosyalari olarak kaydediyoruz. Boylece:
    1) Egitim dongusu cok daha hizli calisir (diskten hazir array okumak,
       NIfTI parse etmekten cok daha ucuz).
    2) Colab'a tasimasi kolaylasir (islenmis .npy'lari yuklemek yeterli).
"""

import os
import glob

import cv2
import numpy as np
import nibabel as nib

# ---- Ayarlar ----
RAW_VOLUME_DIR = "lits_data/volume_pt1"
RAW_SEG_DIR = "lits_data/segmentations"
OUT_IMAGE_DIR = "data/images"
OUT_MASK_DIR = "data/masks"

IMG_SIZE = 256          # egitimde kullanilacak 2D goruntu boyutu
HU_MIN, HU_MAX = -200, 250   # "karaciger penceresi" - bu HU araligi disindaki
                              # dokular (kemik, hava) karaciger ayrimi icin
                              # onemli degil, bu yuzden kirpiyoruz (clipping)


def normalize_hu(slice_2d):
    """CT slice'ini [HU_MIN, HU_MAX] araligina kirpip [0, 1]'e olcekler."""
    slice_2d = np.clip(slice_2d, HU_MIN, HU_MAX)
    slice_2d = (slice_2d - HU_MIN) / (HU_MAX - HU_MIN)
    return slice_2d.astype(np.float32)


def binarize_liver_mask(mask_2d):
    """Label 1 (karaciger) ve 2 (tumor) degerlerini birlestirip
    tek bir 'karaciger var/yok' binary maskesi olusturur."""
    return (mask_2d > 0).astype(np.float32)


def process_volume(volume_path, seg_path, volume_id):
    vol = nib.load(volume_path).get_fdata()   # (H, W, num_slices)
    seg = nib.load(seg_path).get_fdata()

    num_slices = vol.shape[2]
    saved = 0

    for i in range(num_slices):
        img_slice = vol[:, :, i]
        mask_slice = seg[:, :, i]

        img_slice = normalize_hu(img_slice)
        mask_slice = binarize_liver_mask(mask_slice)

        img_slice = cv2.resize(img_slice, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        # Maske icin INTER_NEAREST kullaniyoruz: interpolasyon 0/1 disinda
        # ara degerler (0.3, 0.7 gibi) uretmesin, maske hep 0 veya 1 kalsin.
        mask_slice = cv2.resize(mask_slice, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        out_name = f"vol{volume_id}_slice{i:03d}.npy"
        np.save(os.path.join(OUT_IMAGE_DIR, out_name), img_slice)
        np.save(os.path.join(OUT_MASK_DIR, out_name), mask_slice)
        saved += 1

    return saved


def main():
    os.makedirs(OUT_IMAGE_DIR, exist_ok=True)
    os.makedirs(OUT_MASK_DIR, exist_ok=True)

    volume_paths = sorted(
        glob.glob(os.path.join(RAW_VOLUME_DIR, "volume-*.nii")),
        key=lambda p: int(os.path.basename(p).replace("volume-", "").replace(".nii", "")),
    )

    total_slices = 0
    for vpath in volume_paths:
        volume_id = os.path.basename(vpath).replace("volume-", "").replace(".nii", "")
        spath = os.path.join(RAW_SEG_DIR, f"segmentation-{volume_id}.nii")

        if not os.path.exists(spath):
            print(f"[UYARI] segmentation-{volume_id}.nii bulunamadi, atlaniyor.")
            continue

        n = process_volume(vpath, spath, volume_id)
        total_slices += n
        print(f"volume-{volume_id}: {n} dilim islendi.")

    print(f"\nToplam {total_slices} 2D dilim '{OUT_IMAGE_DIR}' ve '{OUT_MASK_DIR}' altina kaydedildi.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile dataset.py
"""Islenmis 2D dilimler icin PyTorch Dataset ve train/val ayrimi.

ONEMLI: Train/val ayrimini RASTGELE dilim bazinda degil, VOLUME (hasta)
bazinda yapiyoruz. Neden?

Ayni CT taramasindan (ayni hastadan) gelen komsu dilimler birbirine cok
benzer (bitisik kesitler neredeyse ayni goruntu). Eger dilimleri rastgele
karistirip train/val'a bolersek, ayni hastanin bir dilimi train'de, hemen
komsu dilimi val'de olabilir. Bu durumda model val setinde "yeni" bir hasta
gormuyor, aslinda ezberledigi bir hastanin cok benzer bir kesitini goruyor.
Bu da val skorlarinin gercekte oldugundan cok daha iyi cikmasina yol acar
(veri sizintisi / data leakage) ve modelin gercekte ne kadar iyi genelledigi
konusunda bizi yaniltir.

Dogru yontem: hangi hastalarin (volume'larin) train, hangilerinin val
oldugunu ONCE belirleyip, o hastalara ait TUM dilimleri ayni gruba koymak.
"""

import glob
import os

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

IMAGE_DIR = "data/images"
MASK_DIR = "data/masks"


class LiverSliceDataset(Dataset):
    def __init__(self, filenames, image_dir=IMAGE_DIR, mask_dir=MASK_DIR):
        self.filenames = filenames
        self.image_dir = image_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = np.load(os.path.join(self.image_dir, fname))
        mask = np.load(os.path.join(self.mask_dir, fname))

        # PyTorch conv katmanlari [channel, height, width] bekler, numpy'da
        # elimizde sadece [height, width] var - basina kanal boyutu ekliyoruz.
        img = torch.from_numpy(img).unsqueeze(0).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()

        return img, mask


def _volume_id_from_filename(fname):
    # "vol3_slice042.npy" -> "3"
    return fname.split("_")[0].replace("vol", "")


def get_dataloaders(batch_size=8, val_volume_ids=("9", "10"), num_workers=2, pin_memory=False):
    """Tum islemis dosyalari tarar, volume id'sine gore train/val'a boler.

    val_volume_ids: hangi volume'larin dogrulama (validation) icin
    ayrilacagini belirtir. Bu hastalara ait dilimler model egitimi
    sirasinda HIC gorulmez, sadece performansi olcmek icin kullanilir.
    """
    all_files = sorted(os.listdir(IMAGE_DIR))

    train_files = [f for f in all_files if _volume_id_from_filename(f) not in val_volume_ids]
    val_files = [f for f in all_files if _volume_id_from_filename(f) in val_volume_ids]

    train_ds = LiverSliceDataset(train_files)
    val_ds = LiverSliceDataset(val_files)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )

    return train_loader, val_loader


if __name__ == "__main__":
    train_loader, val_loader = get_dataloaders(batch_size=4)
    print(f"Train dilim sayisi: {len(train_loader.dataset)}")
    print(f"Val dilim sayisi  : {len(val_loader.dataset)}")

    imgs, masks = next(iter(train_loader))
    print("Batch goruntu boyutu:", imgs.shape)
    print("Batch maske boyutu  :", masks.shape)
    print("Goruntu deger araligi:", imgs.min().item(), "-", imgs.max().item())
    print("Maske benzersiz degerler:", torch.unique(masks))


In [ ]:
%%writefile train.py
"""U-Net egitim dongusu.

Kullanim:
    python src/train.py --epochs 20 --batch_size 8
    python src/train.py --epochs 1 --limit 100   # hizli sanity check icin
"""

import argparse
import os
import time

import mlflow
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from model import UNet
from losses import DiceBCELoss, iou_score
from dataset import get_dataloaders


def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()  # BatchNorm/Dropout gibi katmanlari "egitim modu"na alir
    running_loss = 0.0

    # tqdm: her batch islendikce ilerleme cubugu ve anlik hizi (it/s) gosterir,
    # boylece egitimin gercekten ilerledigini/ne kadar surdugunu canli goruruz.
    pbar = tqdm(loader, desc="train", leave=False)
    for imgs, masks in pbar:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()          # bir onceki adimin gradyanlarini temizle

        # Mixed precision (AMP): bazi islemleri float16 ile yaparak GPU'da
        # hem hafiza hem hiz kazandirir, dogruluk kaybi ihmal edilebilir
        # duzeydedir. Sadece GPU varsa (cuda) anlamli, CPU'da etkisiz kalir.
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(imgs)      # ileri gecis (forward pass) - tahmin uret
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()  # geri yayilim (backpropagation)
        scaler.step(optimizer)         # agirliklari guncelle
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=loss.item())

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()  # BatchNorm/Dropout "degerlendirme modu"na gecer (farkli davranirlar)
    running_loss = 0.0
    running_iou = 0.0

    with torch.no_grad():  # val sirasinda gradyan hesaplamaya gerek yok, bellek/hiz kazanci
        for imgs, masks in tqdm(loader, desc="val", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, masks)

            running_loss += loss.item() * imgs.size(0)
            running_iou += iou_score(outputs, masks) * imgs.size(0)

    n = len(loader.dataset)
    return running_loss / n, running_iou / n


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--limit", type=int, default=None,
                         help="Sadece ilk N ornekle calis (hizli test icin)")
    parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
    parser.add_argument("--num_workers", type=int, default=None,
                         help="Veri yukleme icin paralel process sayisi. "
                              "Belirtilmezse GPU'da 2, CPU'da 0 kullanilir "
                              "(Windows'ta CPU + coklu worker overhead'i "
                              "faydadan fazla olabiliyor).")
    parser.add_argument("--experiment_name", type=str, default="liver-segmentation",
                         help="MLflow deney (experiment) adi - ayni deney altindaki "
                              "tum run'lar MLflow arayuzunde birlikte listelenir.")
    parser.add_argument("--run_name", type=str, default="unet-baseline",
                         help="Bu calistirmayi tanimlayan isim (orn. 'unet-baseline', "
                              "'segformer-lr1e-4'). MLflow arayuzunde run'lari "
                              "birbirinden ayirt etmek icin kullanilir.")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Cihaz: {device}")

    num_workers = args.num_workers
    if num_workers is None:
        num_workers = 2 if device.type == "cuda" else 0

    train_loader, val_loader = get_dataloaders(
        batch_size=args.batch_size,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )

    if args.limit is not None:
        # Sanity check icin veri setini kucultuyoruz (gercek egitimde kullanilmaz)
        train_loader = DataLoader(
            Subset(train_loader.dataset, range(min(args.limit, len(train_loader.dataset)))),
            batch_size=args.batch_size, shuffle=True,
        )
        val_loader = DataLoader(
            Subset(val_loader.dataset, range(min(args.limit, len(val_loader.dataset)))),
            batch_size=args.batch_size, shuffle=False,
        )

    model = UNet(in_channels=1, out_channels=1).to(device)
    criterion = DiceBCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    # GradScaler: mixed precision sirasinda float16'nin cok kucuk gradyanlari
    # sifira yuvarlamasini (underflow) onlemek icin loss'u gecici olarak
    # buyutup sonra geri kucultur. Sadece GPU'da anlamli.
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    best_iou = 0.0

    # MLflow: ayni deney (experiment) altinda, bu calistirmayi ayri bir "run"
    # olarak kaydediyoruz. set_experiment, boyle bir deney yoksa olusturur,
    # varsa ona baglanir - boylece U-Net ve SegFormer run'lari ayni tabloda
    # yan yana karşılaştırılabilir hale gelir.
    mlflow.set_experiment(args.experiment_name)
    with mlflow.start_run(run_name=args.run_name):
        # log_param: egitim boyunca DEGISMEYEN ayarlar. Bunlari kaydetmezsek,
        # ileride "bu run hangi learning rate ile egitilmisti?" sorusuna
        # cevap veremeyiz.
        mlflow.log_params({
            "model": "UNet",
            "epochs": args.epochs,
            "batch_size": args.batch_size,
            "lr": args.lr,
            "device": device.type,
            "train_samples": len(train_loader.dataset),
            "val_samples": len(val_loader.dataset),
        })

        for epoch in range(1, args.epochs + 1):
            start = time.time()
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
            val_loss, val_iou = validate(model, val_loader, criterion, device)
            elapsed = time.time() - start

            print(f"Epoch {epoch}/{args.epochs} | "
                  f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_IoU={val_iou:.4f} | "
                  f"sure={elapsed:.1f}s")

            # log_metric: DEGISEN degerler, "step" parametresiyle hangi epoch'a
            # ait oldugunu belirtiyoruz. MLflow bunlari otomatik olarak
            # epoch'a karsi bir grafik cizecek sekilde saklar.
            mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_iou": val_iou,
                "epoch_seconds": elapsed,
            }, step=epoch)

            # En iyi modeli sakla (val IoU'ya gore) - her epoch sonunda degil,
            # sadece iyilesme oldugunda kaydederek disk israfini onluyoruz.
            if val_iou > best_iou:
                best_iou = val_iou
                torch.save(model.state_dict(), os.path.join(args.checkpoint_dir, "best_model.pth"))
                print(f"  -> yeni en iyi model kaydedildi (val_IoU={val_iou:.4f})")

        print(f"\nEgitim tamamlandi. En iyi val IoU: {best_iou:.4f}")

        # log_metric (step'siz): run'in NIHAI sonucu, tek bir sayi olarak.
        # MLflow arayuzundeki run listesinde bu sutunu gorup run'lari
        # siralayabiliriz.
        mlflow.log_metric("best_val_iou", best_iou)

        # log_artifact: egitilmis model dosyasini bu run'a kalici olarak
        # ekliyoruz. Ileride "bu run'in modelini indir" dedigimizde buradan gelir.
        mlflow.log_artifact(os.path.join(args.checkpoint_dir, "best_model.pth"))


if __name__ == "__main__":
    main()


## 4. On isleme: NIfTI volume'lari 2D dilimlere ayir

In [ ]:
!python preprocess.py


## 5. (Onerilir) Google Drive'i bagla
Boylece egitim sirasinda kaydedilen en iyi model, Colab oturumu kapansa bile Drive'inda kalir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/u_net_checkpoints


## 6. Egitimi baslat (GPU)
Epoch sayisini ve batch_size'i ihtiyaca gore degistirebilirsin.

In [ ]:
!python train.py --epochs 25 --batch_size 16 --checkpoint_dir /content/drive/MyDrive/u_net_checkpoints


## 7. Tahminleri gorsellestir
Egitilen modelin birkac validation ornegi uzerinde gercek maske ile tahminini yan yana gosterelim (CT dilimi | gercek maske | tahmin).

In [ ]:
%%writefile visualize.py
"""Egitilmis modelin birkac validation ornegi uzerindeki tahminlerini
gorsellestirir: CT dilimi | gercek maske | tahmin maskesi yan yana.

Kullanim (Colab'da):
    python visualize.py --checkpoint /content/drive/MyDrive/u_net_checkpoints/best_model.pth --num_samples 6
"""

import argparse
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from model import UNet
from dataset import get_dataloaders


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--num_samples", type=int, default=6,
                         help="Kac ornek gosterilecek")
    parser.add_argument("--only_with_liver", action="store_true", default=True,
                         help="Sadece gercekte karaciger iceren dilimleri sec "
                              "(bos dilimler gorsel acidan ilginc degil)")
    parser.add_argument("--out_path", type=str, default="results/predictions.png")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = UNet(in_channels=1, out_channels=1).to(device)
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))
    model.eval()

    _, val_loader = get_dataloaders(batch_size=1, num_workers=0)
    val_dataset = val_loader.dataset

    # Sadece karaciger iceren dilimleri bulup rastgele secim yapiyoruz,
    # boylece gorsellerde bos (tamamen arka plan) dilimler yerine anlamli
    # segmentasyon ornekleri goruruz.
    candidate_indices = []
    for i in range(len(val_dataset)):
        _, mask = val_dataset[i]
        if not args.only_with_liver or mask.sum() > 0:
            candidate_indices.append(i)
        if len(candidate_indices) >= args.num_samples * 5:
            break  # tum veri setini taramaya gerek yok, yeterince aday bulunca dur

    chosen = np.random.choice(candidate_indices, size=min(args.num_samples, len(candidate_indices)), replace=False)

    fig, axes = plt.subplots(len(chosen), 3, figsize=(9, 3 * len(chosen)))
    if len(chosen) == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(chosen):
        img, mask = val_dataset[idx]
        img_batch = img.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(img_batch)
            pred = (torch.sigmoid(logits) > 0.5).float()

        img_np = img.squeeze().numpy()
        mask_np = mask.squeeze().numpy()
        pred_np = pred.squeeze().cpu().numpy()

        axes[row, 0].imshow(img_np, cmap="gray")
        axes[row, 0].set_title("CT dilimi")
        axes[row, 1].imshow(mask_np, cmap="gray")
        axes[row, 1].set_title("Gercek maske")
        axes[row, 2].imshow(pred_np, cmap="gray")
        axes[row, 2].set_title("Model tahmini")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    os.makedirs(os.path.dirname(args.out_path), exist_ok=True)
    plt.savefig(args.out_path, dpi=150)
    print(f"Gorsel kaydedildi: {args.out_path}")


if __name__ == "__main__":
    main()


In [ ]:
!python visualize.py --checkpoint /content/drive/MyDrive/u_net_checkpoints/best_model.pth --num_samples 6

from IPython.display import Image, display
display(Image(filename='results/predictions.png'))


## 8. SegFormer ile karsilastirma
Hazir (pretrained) SegFormer-b0 modelini bizim binary karaciger segmentasyonu gorevimize uyarliyoruz: 1 kanal -> 3 kanal, 150 sinif -> 1 sinif, cikti cozunurlugunu geri buyutme. Ayni Dice+BCE loss ve IoU metrigiyle egitip U-Net ile ADIL bir karsilastirma yapacagiz.

In [ ]:
!pip install -q transformers mlflow


In [ ]:
%%writefile segformer_model.py
"""Hazir (pretrained) SegFormer'i bizim binary karaciger segmentasyonu
gorevimize uyarlayan sarmalayici (wrapper).

SegFormer normalde:
  - 3 kanal (RGB) girdi bekler          -> biz 1 kanal (gri) griyi 3'e kopyalayarak cozeriz
  - N farkli sinif icin egitilmistir     -> biz num_labels=1 ile son katmani degistiririz
  - Cikti, girdinin 1/4 cozunurlugunde  -> biz interpolate ile geri buyuturuz

Boylece disaridan bakildiginda, bu model tipki bizim UNet sinifimiz gibi
davranir: [B, 1, H, W] girdi alir, [B, 1, H, W] logit dondurur - ayni
DiceBCELoss ve iou_score fonksiyonlariyla dogrudan kullanilabilir.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"


class SegFormerBinary(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        # ignore_mismatched_sizes=True: HuggingFace'e "son siniflandirma
        # katmaninin boyutu uyusmuyor, onu at ve num_labels'a gore yenisini
        # rastgele baslat" diyoruz. Geri kalan tum agirliklar (encoder)
        # onceden egitilmis haliyle yuklenir.
        self.segformer = SegformerForSemanticSegmentation.from_pretrained(
            model_name,
            num_labels=1,
            ignore_mismatched_sizes=True,
        )

    def forward(self, x):
        # x: [B, 1, H, W] (bizim gri tonlama CT dilimlerimiz)
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)  # [B, 1, H, W] -> [B, 3, H, W]

        target_size = x.shape[2:]  # orijinal H, W - cikisi buna geri buyutecegiz

        outputs = self.segformer(pixel_values=x)
        logits = outputs.logits  # [B, 1, H/4, W/4]

        logits = F.interpolate(logits, size=target_size, mode="bilinear", align_corners=False)
        return logits


if __name__ == "__main__":
    # Sanity check: UNet testimizle birebir ayni sekilde, rastgele bir
    # goruntu ile boyutlarin dogru aktigini dogruluyoruz.
    model = SegFormerBinary()
    dummy_input = torch.randn(2, 1, 256, 256)
    output = model(dummy_input)
    print("Girdi boyutu :", dummy_input.shape)
    print("Cikti boyutu :", output.shape)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Toplam parametre sayisi: {n_params:,}")


In [ ]:
%%writefile train_segformer.py
"""SegFormer fine-tuning dongusu.

train.py ile ayni yapida (ayni loss, metrik, MLflow entegrasyonu) - tek
fark model mimarisi. Boylece U-Net ile SegFormer sonuclari birebir adil
sekilde karsilastirilabilir.

Kullanim:
    python train_segformer.py --epochs 15 --batch_size 8
"""

import argparse
import os
import time

import mlflow
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from segformer_model import SegFormerBinary, MODEL_NAME
from losses import DiceBCELoss, iou_score
from dataset import get_dataloaders


def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc="train", leave=False)
    for imgs, masks in pbar:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=loss.item())

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_iou = 0.0

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="val", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, masks)

            running_loss += loss.item() * imgs.size(0)
            running_iou += iou_score(outputs, masks) * imgs.size(0)

    n = len(loader.dataset)
    return running_loss / n, running_iou / n


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=15,
                         help="U-Net'ten daha az epoch yeterli olabilir - model "
                              "zaten onceden egitilmis (pretrained), sifirdan "
                              "ogrenmiyor, sadece ince ayar (fine-tune) yapiyor.")
    parser.add_argument("--batch_size", type=int, default=8,
                         help="SegFormer, U-Net'ten daha fazla GPU bellegi "
                              "kullanabilir (transformer katmanlari agir), bu "
                              "yuzden varsayilan batch_size'i dusuk tuttuk.")
    parser.add_argument("--lr", type=float, default=6e-5,
                         help="Pretrained modelleri fine-tune ederken genelde "
                              "sifirdan egitimden DAHA DUSUK bir learning rate "
                              "kullanilir - aksi halde onceden ogrenilmis "
                              "faydali agirliklar hizla bozulabilir "
                              "(catastrophic forgetting).")
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
    parser.add_argument("--num_workers", type=int, default=None)
    parser.add_argument("--experiment_name", type=str, default="liver-segmentation")
    parser.add_argument("--run_name", type=str, default="segformer-b0")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Cihaz: {device}")

    num_workers = args.num_workers
    if num_workers is None:
        num_workers = 2 if device.type == "cuda" else 0

    train_loader, val_loader = get_dataloaders(
        batch_size=args.batch_size,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )

    if args.limit is not None:
        train_loader = DataLoader(
            Subset(train_loader.dataset, range(min(args.limit, len(train_loader.dataset)))),
            batch_size=args.batch_size, shuffle=True,
        )
        val_loader = DataLoader(
            Subset(val_loader.dataset, range(min(args.limit, len(val_loader.dataset)))),
            batch_size=args.batch_size, shuffle=False,
        )

    model = SegFormerBinary().to(device)
    criterion = DiceBCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    best_iou = 0.0

    mlflow.set_experiment(args.experiment_name)
    with mlflow.start_run(run_name=args.run_name):
        mlflow.log_params({
            "model": "SegFormer",
            "pretrained_checkpoint": MODEL_NAME,
            "epochs": args.epochs,
            "batch_size": args.batch_size,
            "lr": args.lr,
            "device": device.type,
            "train_samples": len(train_loader.dataset),
            "val_samples": len(val_loader.dataset),
            "total_params": sum(p.numel() for p in model.parameters()),
        })

        for epoch in range(1, args.epochs + 1):
            start = time.time()
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
            val_loss, val_iou = validate(model, val_loader, criterion, device)
            elapsed = time.time() - start

            print(f"Epoch {epoch}/{args.epochs} | "
                  f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_IoU={val_iou:.4f} | "
                  f"sure={elapsed:.1f}s")

            mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_iou": val_iou,
                "epoch_seconds": elapsed,
            }, step=epoch)

            if val_iou > best_iou:
                best_iou = val_iou
                torch.save(model.state_dict(), os.path.join(args.checkpoint_dir, "segformer_best.pth"))
                print(f"  -> yeni en iyi model kaydedildi (val_IoU={val_iou:.4f})")

        print(f"\nEgitim tamamlandi. En iyi val IoU: {best_iou:.4f}")
        mlflow.log_metric("best_val_iou", best_iou)
        mlflow.log_artifact(os.path.join(args.checkpoint_dir, "segformer_best.pth"))


if __name__ == "__main__":
    main()


In [ ]:
!python train_segformer.py --epochs 15 --batch_size 8 --checkpoint_dir /content/drive/MyDrive/u_net_checkpoints


## 9. SegFormer sonuclarini yerel MLflow'a aktar
Yukaridaki hucrenin ciktisinda gorunen her epoch satirini (`Epoch X/15 | train_loss=... | val_loss=... | val_IoU=...`) kopyala, bilgisayarindaki Claude'a yapistir - senin adina `log_segformer_to_mlflow.py` scriptini guncelleyip yerel MLflow'a (U-Net baseline'in yaninda) isleyecek, boylece iki modeli ayni ekranda karsilastirabileceksin.